### Modelo XGBoost, simplificado. El cliente se etiqueta como:

| Etiqueta | Descripción |
|---|---|
| `Confiable` | Buen historial, bajo riesgo |
| `Comportamiento crediticio variable` | Riesgo moderado |
| `Moroso` | Alto riesgo, historial negativo |

In [ ]:
%pip install xgboost scikit-learn pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

print('Dependencias cargadas correctamente.')

### CARGA DE DATOS

In [ ]:
from google.colab import files

print('Suba el archivo clientes_ficticios.csv:')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f'\n Archivo "{filename}" cargado exitosamente.')
print(f'   Dimensiones: {df.shape[0]} clientes × {df.shape[1]} variables')
df.head()

In [ ]:
print('=== Distribución de etiquetas ===')
label_map = {0: 'Confiable', 1: 'Comportamiento crediticio variable', 2: 'Moroso'}
print(df['label'].map(label_map).value_counts())

print('\n=== Estadísticas descriptivas ===')
df.describe().round(2)

### Preparación de datos

In [ ]:
FEATURES = [
    'edad', 'ingreso_mensual', 'antiguedad_laboral_meses',
    'num_productos_banco', 'deuda_total', 'meses_como_cliente',
    'pagos_atrasados_ultimos_12m', 'ratio_deuda_ingreso',
    'score_buro', 'solicitudes_credito_ultimos_6m', 'tiene_garantia'
]
TARGET = 'label'

X = df[FEATURES]
y = df[TARGET]

# Split 80/20 estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Entrenamiento : {X_train.shape[0]} clientes')
print(f'Prueba        : {X_test.shape[0]} clientes')

### Entrenamiento del modelo XGBoost

In [ ]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print('Modelo entrenado.')

In [ ]:
y_pred = model.predict(X_test)

target_names = ['Confiable', 'Comportamiento variable', 'Moroso']
print('=== Reporte de clasificación ===')
print(classification_report(y_test, y_pred, target_names=target_names))

# Matriz de confusión
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de Confusión — Motor de Decisión')
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Importancia de Variables — XGBoost')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

### Motor de Decisión

Aquí es donde se aplican **umbrales de probabilidad** para etiquetar al cliente:

| Condición | Etiqueta |
|---|---|
| P(Confiable) ≥ 0.70 | Confiable |
| P(Moroso) ≥ 0.70 | Moroso |
| Resto | ⚠️ Comportamiento crediticio variable |

In [ ]:
#Función de etiquetado con umbrales
UMBRAL_CONFIABLE = 0.70   # Prob. mínima para etiquetar como confiable
UMBRAL_MOROSO    = 0.70   # Prob. mínima para etiquetar como moroso

def etiquetar_cliente(datos_cliente: dict) -> dict:
    X_nuevo = pd.DataFrame([datos_cliente])[FEATURES]
    probas   = model.predict_proba(X_nuevo)[0]  

    p_confiable = probas[0]
    p_variable  = probas[1]
    p_moroso    = probas[2]

    if p_confiable >= UMBRAL_CONFIABLE:
        etiqueta     = 'Confiable'
        recomendacion = 'Aprobar solicitud. Cliente de bajo riesgo.'
    elif p_moroso >= UMBRAL_MOROSO:
        etiqueta     = 'Moroso'
        recomendacion = 'Rechazar solicitud. Alto riesgo de no pago.'
    else:
        etiqueta     = 'Comportamiento crediticio variable'
        recomendacion = 'Revisar manualmente. Considerar garantías adicionales.'

    return {
        'etiqueta'     : f'{etiqueta}',
        'p_confiable'  : round(p_confiable, 4),
        'p_variable'   : round(p_variable, 4),
        'p_moroso'     : round(p_moroso, 4),
        'recomendacion': recomendacion
    }

print('Motor de decisión listo.')

### Ejemplos

In [ ]:
# Cliente de bajo riesgo
cliente_a = {
    'edad': 45,
    'ingreso_mensual': 3500000,
    'antiguedad_laboral_meses': 90,
    'num_productos_banco': 4,
    'deuda_total': 1200000,
    'meses_como_cliente': 100,
    'pagos_atrasados_ultimos_12m': 0,
    'ratio_deuda_ingreso': 0.03,
    'score_buro': 795,
    'solicitudes_credito_ultimos_6m': 0,
    'tiene_garantia': 1
}

# Cliente de alto riesgo
cliente_b = {
    'edad': 25,
    'ingreso_mensual': 600000,
    'antiguedad_laboral_meses': 5,
    'num_productos_banco': 1,
    'deuda_total': 3200000,
    'meses_como_cliente': 7,
    'pagos_atrasados_ultimos_12m': 4,
    'ratio_deuda_ingreso': 0.75,
    'score_buro': 510,
    'solicitudes_credito_ultimos_6m': 6,
    'tiene_garantia': 0
}

# Comportamiento variable
cliente_c = {
    'edad': 35,
    'ingreso_mensual': 1600000,
    'antiguedad_laboral_meses': 36,
    'num_productos_banco': 2,
    'deuda_total': 2800000,
    'meses_como_cliente': 42,
    'pagos_atrasados_ultimos_12m': 1,
    'ratio_deuda_ingreso': 0.18,
    'score_buro': 660,
    'solicitudes_credito_ultimos_6m': 2,
    'tiene_garantia': 1
}

# Evaluación
for nombre, cliente in [('Cliente A', cliente_a), ('Cliente B', cliente_b), ('Cliente C', cliente_c)]:
    resultado = etiquetar_cliente(cliente)
    print(f'\n{'='*55}')
    print(f'  {nombre}')
    print(f'  Etiqueta      : {resultado["etiqueta"]}')
    print(f'  P(Confiable)  : {resultado["p_confiable"]:.1%}')
    print(f'  P(Variable)   : {resultado["p_variable"]:.1%}')
    print(f'  P(Moroso)     : {resultado["p_moroso"]:.1%}')
    print(f'  Recomendación : {resultado["recomendacion"]}')